# QuantiPhy · IIIXR 2D Static (`SS` + `DS`) · Full 41 câu / 11 video trên GPU T4 ×2

Chạy toàn bộ **41 câu hỏi đo kích thước, khoảng cách hai vật & đường kính quỹ đạo 2D tĩnh** (`12 SS` + `29 DS`) trên 11 video QuantiPhy validation (`internet_0001, internet_0004, internet_0005, internet_0013, internet_0019, internet_0027, simulation_0089, simulation_0098_segmented, simulation_0163, simulation_0193, simulation_0196`):
- **Stage 1 (`Qwen3-VL-8B-Instruct`, shard qua 2×T4 FP16):** Parse fallback + grounding `--notext` cho nhánh `v3nt` (đặc biệt quan trọng cho `#402` trên `simulation_0163` và các mặt trăng Sao Mộc `#498–#509` trên `simulation_0193`), giải phóng toàn bộ VRAM khi xong.
- **Stage 2 (2 worker song song, 1 T4/worker, chia theo video):** GroundingDINO + OWLv2 + SAM 2.1 + Scene Ruler, lưu đầy đủ đa giác đường bao `poly` cho các đối tượng `dist2` để đo khoảng cách cạnh-tới-cạnh chính xác.
- **Stage 3 & Evaluator độc lập:** Đo hình học 2D (`ext_px`, `agg_ext`, `dist2_px` + `_poly_dist`, `orbitdiam`, chuẩn hóa từ kích thước vật chuẩn / tốc độ chuẩn / trọng trường `g = 9.8 m/s²` / `scene_ruler`) + `xground` (`fuse_v3`) trên CPU, kiểm tra chốt chặn 2 bậc độ lớn với `H1 prior`, khóa prediction rồi mới ghép nhãn ground truth, `cached_box` (`preds/val_geo_v4ntrj5sQ2srngsr_d_vw_raw.csv`, tham chiếu MRA `~0.5293`) và `caw` (`quantiphy_full.tar`, tham chiếu MRA `~0.5122`).

Kết quả xuất `/kaggle/working/iiixr_2d_static_full.tar`, `iiixr_2d_static_summary.json`, `per_question.csv` (đủ 41 dòng) và trang phân tích lỗi offline `run/index.html`.

> **Mẹo mở rộng:** Nếu muốn gộp chạy cả 41 câu 2D Static lẫn toàn bộ câu 2D Motion trong cùng một lần trên 12 video 2D (`69` câu), đổi `SCOPE = "all2d_69"` ở ô cấu hình bên dưới.

## 1. Cấu hình

In [ ]:
REPO_URL = "https://github.com/leetranwyng/quantiphy_trannl.git"
SOURCE_REF = "main"  # Có thể pin trực tiếp commit SHA cụ thể
GITHUB_TOKEN_SECRET = "GITHUB_TOKEN"  # Chỉ dùng nếu repo chuyển sang Private

# Để trống để tự tải snapshot đã pin từ HuggingFace; hoặc điền đường dẫn trong /kaggle/input khi chạy offline.
DATASET_DIR = ""
MODELS_DIR = ""
CAW_FULL_TAR = ""  # Để trống để tự tìm quantiphy_full.tar trong /kaggle/input nếu đã gắn
WHEELHOUSE = ""

SCOPE = "static2d_41"      # "static2d_41" (41 câu 2D tĩnh), "static2d_survey158", hoặc "all2d_69" (toàn bộ 69 câu 2D)
DTYPE_PROFILE = "t4_fp16"  # "t4_fp16", "fp32", hoặc "upstream_bf16"
RULE_PARSE_ONLY = False
PARTITION_COUNT = 1        # Tăng >1 nếu muốn chia phiên theo video
PARTITION_INDEX = 0
OUTPUT_BUDGET_GIB = 18.0

## 2. Clone repo tại ref đã chọn & cài đặt dependencies đã khóa

In [ ]:
import base64, os, subprocess, sys, tempfile
from pathlib import Path

source = Path(tempfile.mkdtemp(prefix="iiixr-st2d-")) / "repo"
env = dict(os.environ, GIT_TERMINAL_PROMPT="0")
clone = ["git", "-c", "credential.helper=", "clone", "--quiet", REPO_URL, str(source)]
if subprocess.run(clone, env=env).returncode:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret(GITHUB_TOKEN_SECRET)
    auth = base64.b64encode(("x-access-token:" + token).encode()).decode()
    subprocess.run(clone, check=True, env=dict(env, GIT_CONFIG_COUNT="1",
        GIT_CONFIG_KEY_0="http.https://github.com/.extraHeader",
        GIT_CONFIG_VALUE_0="Authorization: Basic " + auth))
    del token, auth

remote_ref = f"refs/remotes/origin/{SOURCE_REF}"
is_branch = subprocess.run(["git", "show-ref", "--verify", "--quiet", remote_ref], cwd=source).returncode == 0
subprocess.run(["git", "checkout", "--quiet", "--detach", remote_ref if is_branch else SOURCE_REF], cwd=source, check=True)
print("Resolved source commit:", subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=source, text=True).strip())

install = [sys.executable, "-m", "pip", "install", "-q", "-r", str(source / "solutions/iiixr/kaggle/requirements.txt")]
subprocess.run(install + (["--no-index", "--find-links", WHEELHOUSE] if WHEELHOUSE else []), check=True)


## 3. Thực thi Full 41 câu 2D Static & xuất archive + viewer offline

In [ ]:
import runpy
sys.path.insert(0, str(source))
launch = runpy.run_path(str(source / "solutions/iiixr/kaggle/notebook.py"))["launch"]
SUMMARY = launch(source, "full", globals())
